In [2]:
# ============================================================================
# notebook: notebooks/07_robustness.ipynb  (v12 — A' two diagnostics)
# Project: "Incidental vs. Engineered Approval"
# Stage 5a (A'): two-sided replacement bootstrap CIs for the headline numbers.
#   Both groups are resampled with replacement (the correct control that
#   replaces the dropped T2). There is NO composite ensemble; we report the
#   two diagnostics separately, plus a documentation row proving the ensemble
#   has no convergent validity (R-18).
#   Main:
#     Diagnostic 2 — Paradox: density -> default coef (borderline).
#     Diagnostic 1 — Stability group gap (dis - adv).
#   Documentation / appendix:
#     Dropped ensemble V4' coef (FAIL), LowDensity gap, NonFragility gap,
#     and Stability -> default coef (the C7 same-direction paradox).
# Reads results/. Run from notebooks/.
# ---------------------------------------------------------------------------
# PREREQUISITE: run the v12 Stage 3 first so stage3_borderline_scored.parquet
#   carries columns: A_Stability, density_pct, A_LowDensity, A_NonFrag (no ES).
# ============================================================================


# ---------------------------------------------------------------------------
# CELL 1 — Paths, load per-diagnostic borderline set
# ---------------------------------------------------------------------------
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

ROOT    = Path("..").resolve()
RESULTS = ROOT / "results"
RANDOM_STATE = 42
B_BOOT = 2000

B = pd.read_parquet(RESULTS / "stage3_borderline_scored.parquet")
dis = B[B["GROUP"] == "dis_primary"].copy()
adv = B[B["GROUP"] == "advantaged"].copy()
print(f"Borderline: {len(B)}  | dis_primary={len(dis)} adv={len(adv)}")
print("Columns:", [c for c in B.columns if c in
      ("A_Stability","density_pct","A_LowDensity","A_NonFrag")])


# ---------------------------------------------------------------------------
# CELL 2 — Helpers: two-sided bootstrap with percentile CI.
# Each frame is resampled with replacement (both groups for gaps).
# ---------------------------------------------------------------------------
def boot_ci(fn, *frames, B=B_BOOT, seed=RANDOM_STATE):
    """Resample each frame with replacement, apply fn, return (point, lo, hi)."""
    r = np.random.default_rng(seed)
    point = fn(*frames)
    vals = np.empty(B)
    for b in range(B):
        resampled = [f.sample(len(f), replace=True, random_state=int(r.integers(1e9)))
                     for f in frames]
        vals[b] = fn(*resampled)
    lo, hi = np.nanpercentile(vals, [2.5, 97.5])
    return point, lo, hi

def logit_coef(frame, xcol):
    X = sm.add_constant(frame[[xcol, "P_VIP"]])
    m = sm.Logit(frame["DEFAULT"].values, X).fit(disp=0)
    return m.params[xcol]

def gap(a, b, col):
    return a[col].mean() - b[col].mean()

def ens_coef(frame):
    """Dropped ensemble = (Stability + LowDensity)/2 -> default | p(x). R-18 only."""
    e = (frame["A_Stability"] + frame["A_LowDensity"]) / 2.0
    X = sm.add_constant(pd.DataFrame({"_ENS": e.values,
                                      "P_VIP": frame["P_VIP"].values}))
    m = sm.Logit(frame["DEFAULT"].values, X).fit(disp=0)
    return m.params["_ENS"]


# ---------------------------------------------------------------------------
# CELL 3 — Diagnostic 2 (Paradox): density -> default coef, bootstrap CI.
# ---------------------------------------------------------------------------
p, lo, hi = boot_ci(lambda f: logit_coef(f, "density_pct"), B)
print("Diagnostic 2 — Paradox (density->default coef | p(x), borderline):")
print(f"  coef = {p:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  "
      f"{'excludes 0 (paradox robust)' if lo>0 else 'includes 0'}")


# ---------------------------------------------------------------------------
# CELL 4 — Diagnostic 1 (Stability): group gap (dis - adv), bootstrap CI.
# Both groups resampled with replacement (replaces dropped T2).
# ---------------------------------------------------------------------------
p, lo, hi = boot_ci(lambda a, b: gap(a, b, "A_Stability"), dis, adv)
print("Diagnostic 1 — Stability group gap (dis_primary - advantaged):")
print(f"  gap = {p:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  "
      f"{'excludes 0 (robust gap)' if (lo>0 or hi<0) else 'includes 0'}")


# ---------------------------------------------------------------------------
# CELL 5 — Documentation rows (R-18 + C7): why there is no composite.
#   (a) dropped ensemble coef -> no convergent validity.
#   (b) Stability -> default coef: same POSITIVE direction as density (C7),
#       so the two diagnostics push default the SAME way individually but the
#       ensemble mixes Stability(+) with LowDensity(-, the paradox's back side),
#       cancelling out. This is the precise reason the average is invalid.
# ---------------------------------------------------------------------------
pe, elo, ehi = boot_ci(ens_coef, B)
print("\nR-18 — DROPPED ensemble (Stability+LowDensity)/2 -> default:")
print(f"  coef = {pe:+.3f}  95% CI [{elo:+.3f}, {ehi:+.3f}]  "
      f"{'converges' if ehi<0 else 'NO convergence (FAIL) -> dropped'}")

ps, slo, shi = boot_ci(lambda f: logit_coef(f, "A_Stability"), B)
print("\nC7 — Stability -> default coef (same-direction check):")
print(f"  coef = {ps:+.3f}  95% CI [{slo:+.3f}, {shi:+.3f}]  "
      f"{'excludes 0' if (slo>0 or shi<0) else 'includes 0'}")
print("  Both diagnostics relate to default with POSITIVE sign individually;")
print("  the ensemble cancels them (Stability+ vs LowDensity-), hence invalid.")


# ---------------------------------------------------------------------------
# CELL 6 — Appendix gaps (LowDensity, NonFragility), bootstrap CI.
# ---------------------------------------------------------------------------
print("\nAppendix — sub-axis gaps (dis - adv), two-sided bootstrap CI:")
for col in ["A_LowDensity", "A_NonFrag"]:
    p, lo, hi = boot_ci(lambda a, b: gap(a, b, col), dis, adv)
    excl = "excludes 0" if (hi < 0 or lo > 0) else "includes 0"
    note = "(not independent axis)" if col == "A_NonFrag" else "(not significant)"
    print(f"  {col:14} gap={p:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  {excl} {note}")


# ---------------------------------------------------------------------------
# CELL 7 — Assemble MAIN table (two diagnostics) + DOC table + save.
# ---------------------------------------------------------------------------
main_rows = [
    ("Diag2 Paradox (density->default)", *boot_ci(lambda f: logit_coef(f,"density_pct"), B)),
    ("Diag1 Stability gap (dis-adv)",    *boot_ci(lambda a,b: gap(a,b,"A_Stability"), dis, adv)),
]
doc_rows = [
    ("[doc] Dropped ensemble ->default", *boot_ci(ens_coef, B)),
    ("[doc] Stability ->default (C7)",   *boot_ci(lambda f: logit_coef(f,"A_Stability"), B)),
    ("[appx] LowDensity gap",            *boot_ci(lambda a,b: gap(a,b,"A_LowDensity"), dis, adv)),
    ("[appx] NonFragility gap",          *boot_ci(lambda a,b: gap(a,b,"A_NonFrag"), dis, adv)),
]

def show(rows, title):
    tab = pd.DataFrame(rows, columns=["quantity","point","ci_lo","ci_hi"]).round(3)
    tab["excludes_0"] = (tab["ci_hi"] < 0) | (tab["ci_lo"] > 0)
    print(f"\n=== {title} (bootstrap 95% CI) ===")
    print(tab.to_string(index=False))
    return tab

tab_main = show(main_rows, "HEADLINE — TWO DIAGNOSTICS")
tab_doc  = show(doc_rows,  "DOCUMENTATION / APPENDIX")
tab_main.to_csv(RESULTS / "stage5_bootstrap_main_v12.csv", index=False)
tab_doc.to_csv(RESULTS / "stage5_bootstrap_doc_v12.csv", index=False)
print("\nSaved -> stage5_bootstrap_main_v12.csv / stage5_bootstrap_doc_v12.csv")


# ---------------------------------------------------------------------------
# CELL 8 — Robustness verdict
# ---------------------------------------------------------------------------
print("=" * 66)
print("STAGE 5a — BOOTSTRAP ROBUSTNESS VERDICT (v12, A')")
print("=" * 66)
d2 = tab_main[tab_main.quantity.str.startswith("Diag2")].iloc[0]
d1 = tab_main[tab_main.quantity.str.startswith("Diag1")].iloc[0]
ed = tab_doc[tab_doc.quantity.str.contains("ensemble")].iloc[0]
print(f"Diagnostic 2 (Paradox)  : {'robust' if d2.excludes_0 else 'uncertain'}")
print(f"Diagnostic 1 (Stability): {'robust' if d1.excludes_0 else 'uncertain'}")
print(f"Dropped ensemble        : {'FAIL (CI includes 0) -> correctly dropped' if not ed.excludes_0 else 'converges?'}")
print("-" * 66)
print("Interpretation: two independent diagnostics are each robust, but the")
print("composite that averages them has no convergent validity (R-18). Report")
print("the two diagnostics separately; do not report a single reliability score.")
print("This two-sided bootstrap replaces the dropped T2 subsampling control (R-17).")
print("=" * 66)

Borderline: 1141  | dis_primary=254 adv=224
Columns: ['A_Stability', 'density_pct', 'A_LowDensity', 'A_NonFrag']
Diagnostic 2 — Paradox (density->default coef | p(x), borderline):
  coef = +0.976  95% CI [+0.332, +1.639]  excludes 0 (paradox robust)
Diagnostic 1 — Stability group gap (dis_primary - advantaged):
  gap = +0.086  95% CI [+0.045, +0.125]  excludes 0 (robust gap)

R-18 — DROPPED ensemble (Stability+LowDensity)/2 -> default:
  coef = -0.600  95% CI [-1.775, +0.520]  NO convergence (FAIL) -> dropped

C7 — Stability -> default coef (same-direction check):
  coef = +0.965  95% CI [+0.147, +1.692]  excludes 0
  Both diagnostics relate to default with POSITIVE sign individually;
  the ensemble cancels them (Stability+ vs LowDensity-), hence invalid.

Appendix — sub-axis gaps (dis - adv), two-sided bootstrap CI:
  A_LowDensity   gap=-0.033  95% CI [-0.083, +0.019]  includes 0 (not significant)
  A_NonFrag      gap=-0.072  95% CI [-0.110, -0.033]  excludes 0 (not independent axis)
